# Smart Parking · Entraînement YOLO v1 (sprint 4)

Entraîne **YOLO11n** à détecter les voitures (une seule classe, `voiture`) sur le dataset du projet :
lots 1, 2 et 3 réunis puis découpés en entraînement (70 %), validation (15 %) et test (15 %).

**Sur Kaggle** : GPU activé, dataset `dataset_smartparking_s4.zip` **privé** ajouté en entrée, puis *Run All*.
Les fichiers à télécharger se trouvent à la fin dans `/kaggle/working/yolo_v1/` (onglet *Output*).

Procédure complète : `iot/entrainement/README.md` dans le dépôt.

## 1. Hyperparamètres
Tout ce qu'on peut changer est ici.

In [ ]:
import os

MODELE_DEPART = "yolo11n.pt"   # YOLO11n pré-entraîné sur COCO (téléchargé automatiquement)
EPOCHS = int(os.environ.get("EPOCHS", 50))   # nombre maximal de passages sur le dataset
PATIENCE = 10                  # arrêt anticipé si la validation ne s'améliore plus pendant 10 epochs
IMGSZ = 640                    # taille des images à l'entraînement (comme sur le Pi)
BATCH = 16                     # images par lot ; baisser à 8 en cas de mémoire GPU insuffisante
SEED = 42                      # même graine = même entraînement
DEVICE = os.environ.get("DEVICE", 0)   # 0 = premier GPU de Kaggle
NOM = "yolo_v1"

# Dossiers de Kaggle (modifiables par variables d'environnement, pour un essai local)
ENTREE = os.environ.get("KAGGLE_INPUT", "/kaggle/input")      # datasets ajoutés en entrée (lecture seule)
TRAVAIL = os.environ.get("KAGGLE_WORKING", "/kaggle/working")  # dossier de travail (sauvegardé dans Output)

## 2. Vérifier le GPU
Si aucun GPU n'apparaît : *Settings* → *Accelerator* → **GPU T4 x2** (ou P100), puis relancer.

In [ ]:
!nvidia-smi
import torch
print("GPU disponible :", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "aucun")

## 3. Installer Ultralytics
Même contrainte de version que `iot/requirements.txt` (`ultralytics>=8.3`).

In [ ]:
%pip install -q "ultralytics>=8.3"
import ultralytics
print("Ultralytics", ultralytics.__version__)

## 4. Trouver le dataset et le copier
Kaggle place le dataset ajouté en entrée dans `/kaggle/input/<nom>/`, **en lecture seule**.
On cherche le dossier qui contient `data.yaml`, `entrainement/`, `validation/` et `test/`, on le copie dans
`/kaggle/working/dataset/`, puis on réécrit `data.yaml` avec des chemins absolus.

In [ ]:
import shutil
from pathlib import Path

def trouver_dataset(racine):
    """Le dossier qui contient data.yaml et les trois parties (quelle que soit la profondeur)."""
    for yaml in sorted(Path(racine).rglob("data.yaml")):
        dossier = yaml.parent
        if all((dossier / p / "images").is_dir() for p in ("entrainement", "validation", "test")):
            return dossier
    raise FileNotFoundError(f"Aucun dataset trouvé dans {racine} : ajouter dataset_smartparking_s4 en entrée (Add Input).")

source = trouver_dataset(ENTREE)
DATASET = Path(TRAVAIL) / "dataset"
if DATASET.exists():
    shutil.rmtree(DATASET)
shutil.copytree(source, DATASET)

DATA_YAML = DATASET / "data.yaml"
DATA_YAML.write_text(
    f"path: {DATASET.as_posix()}\n"
    "train: entrainement/images\n"
    "val: validation/images\n"
    "test: test/images\n"
    "nc: 1\n"
    "names: ['voiture']\n",
    encoding="utf-8",
)
print("Dataset trouvé :", source)
for partie in ("entrainement", "validation", "test"):
    print(f"  {partie:12s} {len(list((DATASET / partie / 'images').iterdir())):4d} images")
print(DATA_YAML.read_text(encoding="utf-8"))

## 5. Entraîner YOLO11n
Durée indicative sur un GPU T4 : quelques minutes pour 50 epochs. Le meilleur modèle (sur la validation) est gardé dans `best.pt`.

In [ ]:
from ultralytics import YOLO

modele = YOLO(MODELE_DEPART)
entrainement = modele.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    deterministic=True,
    device=DEVICE,
    project=str(Path(TRAVAIL) / "runs"),
    name=NOM,
    exist_ok=True,
    plots=True,
)
DOSSIER_RUN = Path(entrainement.save_dir)
BEST = DOSSIER_RUN / "weights" / "best.pt"
print("Meilleur modèle :", BEST)

## 6. Courbes et matrice de confusion
`results.png` : pertes et métriques à chaque epoch (une validation qui remonte alors que l'entraînement baisse = sur-apprentissage).

In [ ]:
from IPython.display import Image, display

for nom in ("results.png", "confusion_matrix.png"):
    chemin = DOSSIER_RUN / nom
    if chemin.exists():
        print(nom)
        display(Image(filename=str(chemin), width=900))

## 7. Métriques sur la validation
Le meilleur modèle est évalué sur la **validation** (les images qui ont servi à le choisir).

In [ ]:
import json

def metriques(resultat):
    b = resultat.box
    return {"mAP50": round(float(b.map50), 4), "mAP50-95": round(float(b.map), 4),
            "precision": round(float(b.mp), 4), "rappel": round(float(b.mr), 4)}

meilleur = YOLO(str(BEST))
val = metriques(meilleur.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, device=DEVICE,
                             project=str(Path(TRAVAIL) / "runs"), name=NOM + "_val", exist_ok=True))
print("Validation :", json.dumps(val, indent=2))

## 8. Évaluation finale sur le test (une seule fois)
Le **test** n'a jamais servi pendant l'entraînement ni pour choisir le modèle : c'est la mesure honnête de ses performances.
On ne l'exécute **qu'une fois**, à la fin ; on ne modifie plus rien en fonction de ce résultat dans cette version.

In [ ]:
test = metriques(meilleur.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, device=DEVICE,
                              project=str(Path(TRAVAIL) / "runs"), name=NOM + "_test", exist_ok=True))
print("Test :", json.dumps(test, indent=2))

## 9. Fichiers à télécharger
Tout est copié dans `/kaggle/working/yolo_v1/`, à récupérer dans l'onglet **Output**.

In [ ]:
SORTIE = Path(TRAVAIL) / NOM
SORTIE.mkdir(parents=True, exist_ok=True)
for nom in ("weights/best.pt", "results.csv", "results.png", "confusion_matrix.png"):
    shutil.copy2(DOSSIER_RUN / nom, SORTIE / Path(nom).name)
(SORTIE / "metriques.json").write_text(json.dumps({
    "modele_depart": MODELE_DEPART, "epochs_max": EPOCHS, "patience": PATIENCE, "imgsz": IMGSZ,
    "batch": BATCH, "seed": SEED, "ultralytics": ultralytics.__version__,
    "validation": val, "test": test,
}, indent=2, ensure_ascii=False), encoding="utf-8")
for f in sorted(SORTIE.iterdir()):
    print(f"{f.name:22s} {f.stat().st_size / 1e6:6.2f} Mo")

## 10. Nettoyage (licences du dataset)
Tout ce qui reste dans `/kaggle/working/` est enregistré dans la version du notebook. Le dataset contient des images
**CARPK (usage académique uniquement)** et des photos sous droits d'auteur : on supprime la copie du dataset et les
dossiers d'entraînement, qui contiennent des images d'exemple. Seul `yolo_v1/` reste.

In [ ]:
for dossier in (DATASET, Path(TRAVAIL) / "runs"):
    shutil.rmtree(dossier, ignore_errors=True)
print("Reste dans", TRAVAIL, ":", sorted(p.name for p in Path(TRAVAIL).iterdir()))